In [1]:
from pydantic import BaseModel, Field
from typing import List, Optional
import json
import os
import sys
from mistralai import Mistral
sys.path.append(os.getcwd())
from retrieval_app.core import MISTRAL_MODEL, mistral_api_key, MODEL_BACKEND, COHERE_MODEL, cohere_api_key
import numpy as np
import os
from pathlib import Path
path_to_data = os.path.join(os.getcwd(), "data")
path_to_corpus = os.path.join(os.getcwd(), "data", "corpus")
path_to_corpus_splitted = os.path.join(path_to_data, "corpus_splitted_multihop")
path_to_debat = os.path.join(path_to_corpus_splitted,"1881-01-20")

/home/atom/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


##

## Import corpus

In [2]:
files = sorted(os.listdir(path_to_debat))
corpus = []
file_to_content = {}  # Dictionary mapping file names to their content

for f in sorted(files):
    with open(os.path.join(path_to_debat, f), 'r', encoding='utf-8') as file:
        text = file.read()
        corpus.append(text)
        file_to_content[f] = text  # Store mapping

print(f"Created file_to_content dictionary with {len(file_to_content)} files")

Created file_to_content dictionary with 14 files


## Prepare multi gop

In [9]:
MULTI_HOP_SYSTEM_PROMPT_OLD = """Vous êtes un expert dans la création de questions multi-hop nécessitant un raisonnement à travers plusieurs sources textuelles dans le contexte de l'analyse des débats parlementaires de la Troisième République française.

Vous allez recevoir plusieurs extraits (chunks) d'un débat parlementaire issus de fichiers dont l'identifiant est fourni. Votre tâche est de générer {num_questions} questions multi-hop basées sur ces extraits.

EXIGENCE ABSOLUE : Chaque question générée DOIT nécessiter des informations d'au moins 2 fichiers différents pour y répondre complètement. Les questions utilisant un seul fichier sont INTERDITES.

Chaque question doit :
1. **OBLIGATOIREMENT** nécessiter des informations d'au moins 2 extraits différents pour y répondre complètement
2. Impliquer une forme de raisonnement, de comparaison ou de synthèse entre les fichiers
3. Être répondable à partir des extraits de texte donnés
4. Porter sur des éléments factuels des débats parlementaires
5. Donner suffisamment de contexte pour permettre une réponse complète
6. Avoir des étapes de raisonnement claires qui montrent comment croiser les informations des différents fichiers

Types de questions à privilégier dans le contexte parlementaire :
- Comparative : Comparer les positions ou arguments de différents députés/orateurs présents dans différents fichiers
- Causale : Établir des relations de cause à effet entre événements mentionnés dans différents fichiers
- Séquentielle : Questions sur les procédures parlementaires nécessitant l'ordre chronologique entre fichiers
- Analytique : Questions nécessitant la synthèse de plusieurs faits ou arguments parlementaires de fichiers distincts
- Factuelle : Questions sur des faits précis nécessitant de croiser des informations de plusieurs fichiers

RAPPEL CRITIQUE : Toute question nécessitant seulement un fichier sera rejetée. Vous devez absolument créer des liens entre les fichiers.
"""

MULTI_HOP_SYSTEM_PROMPT = """
Vous êtes un expert en génération de questions complexes (multi-hop) dans le cadre de l’analyse des débats parlementaires de la Troisième République française.

Votre tâche est de générer exactement {num_questions} questions multi-hop à partir d’un ensemble d’extraits (chunks) issus de fichiers de débats parlementaires. Chaque extrait est associé à un identifiant de fichier, mais **les questions générées ne doivent en aucun cas contenir ces noms ou identifiants**.

### DÉFINITION
Une **question multi-hop** est une question nécessitant de croiser des informations provenant d’au **moins deux extraits différents**, chacun issu de **fichiers distincts**, afin de produire une réponse complète et fondée.

### CONTRAINTES OBLIGATOIRES
1. **Aucune question ne doit être générée si les conditions suivantes ne peuvent pas être rigoureusement respectées.**
2. Chaque question DOIT s'appuyer sur des extraits provenant d’au moins 2 fichiers différents.
3. Les questions doivent :
   - Être basées uniquement sur des extraits ayant une **thématique commune identifiable** (ex. : politique coloniale, budget de l’armée, réforme scolaire, etc.)
   - Nécessiter un **raisonnement croisé, comparatif ou synthétique** entre les extraits
   - Être **répondables uniquement à partir du contenu fourni**
   - Ne **pas contenir de noms de fichiers ou d’identifiants techniques**
   - Être **factuelles, précises et ancrées dans le contenu des débats**

4. Toute question se basant sur un seul fichier ou sur des extraits sans lien thématique clair doit être **strictement rejetée**.

### TYPES DE QUESTIONS À PRIVILÉGIER
- **Comparative** : Compare des positions ou arguments de différents orateurs apparaissant dans des fichiers distincts mais sur un même sujet
- **Causale** : Met en relation des causes et conséquences évoquées dans des extraits différents mais thématiquement liés
- **Séquentielle** : Nécessite de comprendre l’ordre ou l’évolution d’un débat ou d’un processus parlementaire à travers plusieurs fichiers
- **Analytique** : Synthétise plusieurs arguments convergents ou contradictoires autour d’une même thématique
- **Factuelle croisée** : Interroge un fait qui ne peut être établi qu’en croisant deux sources distinctes sur un même sujet

### STRUCTURE ATTENDUE (conformément au schéma JSON suivant)
Pour chaque question générée, fournir :
- `question` : la formulation complète (ne mentionne aucun identifiant de fichier)
- `etapes_raisonnement` : liste d’étapes montrant comment les informations de plusieurs extraits sont croisées pour répondre
- `fichiers_requis` : noms des fichiers utilisés (au moins 2)
- `difficulte` : facile | moyen | difficile
- `type_question` : factuelle | analytique | comparative | causale | sequentielle

Un champ `resume_sources` doit également résumer brièvement les thématiques abordées dans les extraits et indiquer les principales connexions ou chevauchements thématiques détectés entre les fichiers.

⚠️ **IMPORTANT :**
- Si aucune combinaison d’extraits ne permet de créer une question respectant les critères ci-dessus, **vous ne devez générer aucune question.** Le champ `questions` doit alors être une liste vide.
"""



class QuestionMultiHop(BaseModel):
    question: str = Field(description="La question multi-hop qui DOIT obligatoirement nécessiter des informations croisées d'au moins 2 fichiers de débats parlementaires différents")
    etapes_raisonnement: List[str] = Field(description="Liste des étapes de raisonnement montrant explicitement comment croiser les informations de plusieurs fichiers pour répondre à la question")
    fichiers_requis: List[str] = Field(description="Noms des fichiers de débats parlementaires nécessaires (MINIMUM 2 fichiers obligatoires)")
    difficulte: str = Field(description="Niveau de difficulté de la synthèse multi-fichiers : facile, moyen, difficile")
    type_question: str = Field(description="Type de question multi-hop : factuelle, analytique, comparative, causale, sequentielle")

class EnsembleQuestionsMultiHop(BaseModel):
    questions: List[QuestionMultiHop] = Field(description="Liste des questions multi-hop générées, chacune nécessitant obligatoirement au moins 2 fichiers de débats parlementaires")
    resume_sources: str = Field(description="Résumé succinct du contenu des différents fichiers de débats parlementaires analysés et de leurs interconnexions")

In [ ]:
def generate_multihop_questions_from_files(file_names, model_service_func, num_questions=3):
    """
    Generate multihop questions from file names directly.
    
    Args:
        file_names: List of file names to analyze
        model_service_func: Function to call the LLM
        num_questions: Number of questions to generate
    
    Returns:
        Response from the LLM with generated questions
    """
    # Extract chunks from file names
    text_chunks = []
    valid_files = []
    
    for file_name in file_names:
        if file_name in file_to_content:
            text_chunks.append(file_to_content[file_name])
            valid_files.append(file_name)
        else:
            print(f"Warning: File {file_name} not found in corpus")
    
    if not text_chunks:
        print("Error: No valid files found")
        return None
    
    # Prepare the prompt for multihop question generation with file names
    chunks_text = "\n\n".join([f"Fichier {valid_files[i]}: {chunk[:500]}..." if len(chunk) > 500 else f"Fichier {valid_files[i]}: {chunk}" 
                              for i, chunk in enumerate(text_chunks)])
    
    messages = [
        {
            "role": "system", 
            "content": MULTI_HOP_SYSTEM_PROMPT.format(num_questions=num_questions)
        },
        {
            "role": "user",
            "content": f"Voici les extraits de débats parlementaires à analyser :\n\n{chunks_text}\n\nGénérez des questions multi-hop basées sur ces extraits. RAPPEL CRITIQUE : Chaque question DOIT obligatoirement nécessiter des informations d'au moins 2 fichiers différents. Utilisez les noms de fichiers dans le champ fichiers_requis."
        }
    ]
    try:
        response = model_service_func(messages, EnsembleQuestionsMultiHop, temperature=0.3)
        return response
    except Exception as e:
        print(f"Error generating multihop questions: {str(e)}")
        return None


Loaded 14 text chunks from the debate corpus
Available files: ['1881-01-20_000.txt', '1881-01-20_001.txt', '1881-01-20_002.txt', '1881-01-20_003.txt', '1881-01-20_004.txt']...


In [12]:
def get_mistral_structured_response(messages, response_format,temperature=0.3):
    try:
        client = Mistral(api_key=mistral_api_key)
        response = client.chat.parse(
            model=MISTRAL_MODEL,
            messages=messages,
            response_format=response_format
        )
        return response.choices[0]
    except Exception as e:
        raise Exception(f"Error in Mistral structured output response: {str(e)}")

def display_multihop_questions(multihop_response):
    """
    Display multihop questions in a nice formatted way.
    
    Args:
        multihop_response: Response from the LLM containing multihop questions
    """
    if not multihop_response or not hasattr(multihop_response, 'message'):
        print("❌ Aucune question générée")
        return
    
    questions_data = multihop_response.message.parsed
    
    print("=" * 80)
    print("🎯 QUESTIONS MULTI-HOP GÉNÉRÉES")
    print("=" * 80)
    
    # Display source summary
    if hasattr(questions_data, 'resume_sources') and questions_data.resume_sources:
        print(f"\n📝 Résumé des sources :")
        print(f"   {questions_data.resume_sources}")
    
    print(f"\n📊 Nombre de questions générées : {len(questions_data.questions)}")
    
    # Display each question
    for i, question in enumerate(questions_data.questions, 1):
        print(f"\n" + "─" * 60)
        print(f"❓ QUESTION {i}")
        print("─" * 60)
        
        # Question details
        print(f"🔍 Type : {question.type_question.upper()}")
        print(f"⚡ Difficulté : {question.difficulte.upper()}")
        print(f"📁 Fichiers requis : {', '.join(question.fichiers_requis)}")
        
        print(f"\n💭 Question :")
        print(f"   {question.question}")
        
        print(f"\n🧠 Étapes de raisonnement :")
        for j, step in enumerate(question.etapes_raisonnement, 1):
            print(f"   {j}. {step}")
    
    print("\n" + "=" * 80)

## Run multi hop

In [15]:
file_to_content.keys()  # Ensure the dictionary is loaded

dict_keys(['1881-01-20_000.txt', '1881-01-20_001.txt', '1881-01-20_002.txt', '1881-01-20_003.txt', '1881-01-20_004.txt', '1881-01-20_005.txt', '1881-01-20_006.txt', '1881-01-20_007.txt', '1881-01-20_008.txt', '1881-01-20_009.txt', '1881-01-20_010.txt', '1881-01-20_011.txt', '1881-01-20_012.txt', '1881-01-20_013.txt'])

In [17]:
# Now you can simply provide file names
selected_file_names = ['1881-01-20_000.txt', '1881-01-20_012.txt']

print("🚀 Génération des questions multi-hop...")
print(f"📂 Fichiers sélectionnés : {selected_file_names}")

multihop_questions = generate_multihop_questions_from_files(
    file_names=selected_file_names,
    model_service_func=get_mistral_structured_response,
    num_questions=3
)

🚀 Génération des questions multi-hop...
📂 Fichiers sélectionnés : ['1881-01-20_000.txt', '1881-01-20_012.txt']


In [18]:
display_multihop_questions(multihop_questions)

🎯 QUESTIONS MULTI-HOP GÉNÉRÉES

📝 Résumé des sources :
   Les extraits fournis sont insuffisants pour générer des questions multi-hop valides. Le fichier '1881-01-20_000.txt' est vide et ne contient aucune information exploitable. Le fichier '1881-01-20_012.txt' contient des informations sur une succession et des créanciers, mais il n'y a pas de second fichier avec des informations complémentaires permettant de croiser les données pour formuler une question multi-hop.

📊 Nombre de questions générées : 0



## Génération en lot - Pairs de fichiers adjacents

In [17]:
import json
from itertools import combinations
from datetime import datetime

def generate_batch_multihop_questions(output_file="multihop_questions.jsonl", num_questions_per_pair=3):
    """
    Generate multihop questions for all adjacent pairs of files in the corpus
    and save them to a JSONL file. Each result is saved immediately after generation.
    
    Args:
        output_file: Path to the output JSONL file
        num_questions_per_pair: Number of questions to generate per file pair
    """
    
    # Get all file names and sort them
    all_files = sorted(list(file_to_content.keys()))
    print(f"📂 Total files in corpus: {len(all_files)}")
    
    # Create pairs of adjacent files (side by side)
    file_pairs = []
    for i in range(len(all_files) - 1):
        file_pairs.append([all_files[i], all_files[i + 1]])
    
    print(f"👥 Total adjacent pairs to process: {len(file_pairs)}")
    
    # Prepare output file path and initialize counters
    output_path = os.path.join(os.getcwd(), "data", output_file)
    successful_generations = 0
    failed_generations = 0
    all_results = []  # Keep for return value and summary
    
    # Initialize/clear the output file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("")  # Clear the file
    
    print(f"📄 Output file initialized: {output_path}")
    print(f"💾 Results will be saved immediately after each generation")
    
    # Process each pair
    for i, pair in enumerate(file_pairs):
        print(f"\n🔄 Processing pair {i+1}/{len(file_pairs)}: {pair}")
        
        try:
            # Generate questions for this pair
            multihop_response = generate_multihop_questions_from_files(
                file_names=pair,
                model_service_func=get_mistral_structured_response,
                num_questions=num_questions_per_pair
            )
            
            if multihop_response and hasattr(multihop_response, 'message'):
                questions_data = multihop_response.message.parsed
                
                # Create entry for this pair
                pair_result = {
                    "pair_id": i + 1,
                    "file_pair": pair,
                    "timestamp": datetime.now().isoformat(),
                    "num_questions": len(questions_data.questions),
                    "resume_sources": questions_data.resume_sources,
                    "questions": []
                }
                
                # Add each question
                for j, question in enumerate(questions_data.questions):
                    question_dict = {
                        "question_id": f"pair_{i+1}_q_{j+1}",
                        "question": question.question,
                        "etapes_raisonnement": question.etapes_raisonnement,
                        "fichiers_requis": question.fichiers_requis,
                        "difficulte": question.difficulte,
                        "type_question": question.type_question
                    }
                    pair_result["questions"].append(question_dict)
                
                # SAVE IMMEDIATELY after successful generation
                with open(output_path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(pair_result, ensure_ascii=False) + '\n')
                
                all_results.append(pair_result)
                successful_generations += 1
                
                print(f"   ✅ Successfully generated {len(questions_data.questions)} questions")
                print(f"   💾 Saved to file immediately")
                
            else:
                print(f"   ❌ Failed to generate questions for pair {pair}")
                failed_generations += 1
                
        except Exception as e:
            print(f"   ❌ Error processing pair {pair}: {str(e)}")
            failed_generations += 1
            continue
    
    # Summary
    print(f"\n{'='*60}")
    print(f"📊 RÉSUMÉ DE LA GÉNÉRATION EN LOT")
    print(f"{'='*60}")
    print(f"📂 Fichier de sortie: {output_path}")
    print(f"👥 Paires traitées avec succès: {successful_generations}")
    print(f"❌ Paires échouées: {failed_generations}")
    total_questions = sum(len(r['questions']) for r in all_results)
    print(f"📝 Total questions générées: {total_questions}")
    if (successful_generations + failed_generations) > 0:
        print(f"📈 Taux de réussite: {(successful_generations/(successful_generations+failed_generations)*100):.1f}%")
    print(f"💾 Toutes les questions ont été sauvegardées au fur et à mesure")
    
    return all_results

In [18]:
# Execute the batch generation
print("🚀 LANCEMENT DE LA GÉNÉRATION EN LOT")
print("⚠️  Cela peut prendre plusieurs minutes selon le nombre de fichiers...")

# Run the batch generation (you can adjust the number of questions per pair)
batch_results = generate_batch_multihop_questions(
    output_file="multihop_questions_adjacent_pairs.jsonl",
    num_questions_per_pair=2  # Adjust this number as needed
)

🚀 LANCEMENT DE LA GÉNÉRATION EN LOT
⚠️  Cela peut prendre plusieurs minutes selon le nombre de fichiers...
📂 Total files in corpus: 14
👥 Total adjacent pairs to process: 13
📄 Output file initialized: /home/atom/Bureau/Aurelien/RAGbattre/data/multihop_questions_adjacent_pairs.jsonl
💾 Results will be saved immediately after each generation

🔄 Processing pair 1/13: ['1881-01-20_000.txt', '1881-01-20_001.txt']
   ✅ Successfully generated 0 questions
   💾 Saved to file immediately

🔄 Processing pair 2/13: ['1881-01-20_001.txt', '1881-01-20_002.txt']
   ✅ Successfully generated 0 questions
   💾 Saved to file immediately

🔄 Processing pair 2/13: ['1881-01-20_001.txt', '1881-01-20_002.txt']
   ✅ Successfully generated 0 questions
   💾 Saved to file immediately

🔄 Processing pair 3/13: ['1881-01-20_002.txt', '1881-01-20_003.txt']
   ✅ Successfully generated 0 questions
   💾 Saved to file immediately

🔄 Processing pair 3/13: ['1881-01-20_002.txt', '1881-01-20_003.txt']
   ✅ Successfully generate

In [19]:
def load_and_display_jsonl_results(jsonl_file="multihop_questions.jsonl", max_pairs_to_show=3):
    """
    Load and display results from the JSONL file.
    
    Args:
        jsonl_file: Path to the JSONL file
        max_pairs_to_show: Maximum number of pairs to display in detail
    """
    
    try:
        file_path = os.path.join(os.getcwd(),"data", jsonl_file)
        
        if not os.path.exists(file_path):
            print(f"❌ File {file_path} not found")
            return
        
        # Load all results
        results = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                results.append(json.loads(line.strip()))
        
        print(f"📂 Chargé {len(results)} paires depuis {jsonl_file}")
        print(f"📊 Total questions: {sum(len(r['questions']) for r in results)}")
        
        # Display detailed results for first few pairs
        for i, result in enumerate(results[:max_pairs_to_show]):
            print(f"\n{'='*50}")
            print(f"📋 PAIRE {result['pair_id']}: {' & '.join(result['file_pair'])}")
            print(f"{'='*50}")
            print(f"🕒 Timestamp: {result['timestamp']}")
            print(f"📝 Nombre de questions: {result['num_questions']}")
            
            if result.get('resume_sources'):
                print(f"\n📄 Résumé des sources:")
                print(f"   {result['resume_sources']}")
            
            # Show questions
            for j, question in enumerate(result['questions']):
                print(f"\n❓ Question {j+1} ({question['type_question']} - {question['difficulte']}):")
                print(f"   {question['question']}")
                print(f"📁 Fichiers requis: {', '.join(question['fichiers_requis'])}")
        
        if len(results) > max_pairs_to_show:
            print(f"\n... et {len(results) - max_pairs_to_show} autres paires")
        
        return results
        
    except Exception as e:
        print(f"❌ Erreur lors du chargement: {str(e)}")
        return None

In [21]:
# Load and display the generated results
print("📖 CHARGEMENT ET AFFICHAGE DES RÉSULTATS")

loaded_results = load_and_display_jsonl_results(
    jsonl_file="multihop_questions_adjacent_pairs.jsonl",
    max_pairs_to_show=15  # Show details for first 3 pairs
)

📖 CHARGEMENT ET AFFICHAGE DES RÉSULTATS
📂 Chargé 13 paires depuis multihop_questions_adjacent_pairs.jsonl
📊 Total questions: 11

📋 PAIRE 1: 1881-01-20_000.txt & 1881-01-20_001.txt
🕒 Timestamp: 2025-06-18T11:00:58.487622
📝 Nombre de questions: 0

📄 Résumé des sources:
   Les extraits fournis ne contiennent pas suffisamment d'informations thématiques communes pour générer des questions multi-hop conformes aux critères spécifiés. Le premier fichier est vide, et le second ne contient que des informations de session sans contenu de débat substantiel.

📋 PAIRE 2: 1881-01-20_001.txt & 1881-01-20_002.txt
🕒 Timestamp: 2025-06-18T11:01:07.476657
📝 Nombre de questions: 0

📄 Résumé des sources:
   Les extraits fournis concernent principalement des aspects procéduraux et organisationnels des séances parlementaires, incluant des incidents, des communications officielles, des excuses et demandes de congés, ainsi que des processus de nomination de divers postes parlementaires. Il n'y a pas de thématiq

## Formats de visualisation élégants

In [22]:
def convert_jsonl_to_readable_text(jsonl_file="multihop_questions_adjacent_pairs.jsonl", 
                                   output_file="questions_multihop_readable.txt"):
    """
    Convertit le fichier JSONL en un format texte lisible et élégant.
    
    Args:
        jsonl_file: Fichier JSONL source
        output_file: Fichier texte de sortie
    """
    import json
    from datetime import datetime
    
    try:
        input_path = os.path.join(os.getcwd(), "data", jsonl_file)
        output_path = os.path.join(os.getcwd(), "data", output_file)
        
        if not os.path.exists(input_path):
            print(f"❌ Fichier {input_path} non trouvé")
            return
        
        # Charger les données
        results = []
        with open(input_path, 'r', encoding='utf-8') as f:
            for line in f:
                results.append(json.loads(line.strip()))
        
        # Créer le fichier texte élégant
        with open(output_path, 'w', encoding='utf-8') as f:
            # En-tête du document
            f.write("="*80 + "\n")
            f.write("🎯 QUESTIONS MULTI-HOP GÉNÉRÉES - DÉBATS PARLEMENTAIRES\n")
            f.write("="*80 + "\n")
            f.write(f"📅 Généré le: {datetime.now().strftime('%d/%m/%Y à %H:%M:%S')}\n")
            f.write(f"📊 Nombre total de paires: {len(results)}\n")
            total_questions = sum(len(r['questions']) for r in results)
            f.write(f"📝 Nombre total de questions: {total_questions}\n")
            f.write("="*80 + "\n\n")
            
            # Pour chaque paire
            for i, result in enumerate(results, 1):
                f.write(f"{'='*60}\n")
                f.write(f"📋 PAIRE {result['pair_id']}: {' & '.join(result['file_pair'])}\n")
                f.write(f"{'='*60}\n")
                f.write(f"🕒 Généré le: {result['timestamp'][:19].replace('T', ' à ')}\n")
                f.write(f"📝 Nombre de questions: {result['num_questions']}\n\n")
                
                # Résumé des sources
                if result.get('resume_sources'):
                    f.write("📄 RÉSUMÉ DES SOURCES:\n")
                    f.write(f"{result['resume_sources']}\n\n")
                
                # Questions
                if result['questions']:
                    f.write("❓ QUESTIONS GÉNÉRÉES:\n")
                    f.write("-"*40 + "\n")
                    
                    for j, question in enumerate(result['questions'], 1):
                        f.write(f"\n🔹 QUESTION {j}\n")
                        f.write(f"   Type: {question['type_question'].upper()}\n")
                        f.write(f"   Difficulté: {question['difficulte'].upper()}\n")
                        f.write(f"   Fichiers requis: {', '.join(question['fichiers_requis'])}\n\n")
                        
                        # Question principale
                        f.write("   💭 QUESTION:\n")
                        f.write(f"   {question['question']}\n\n")
                        
                        # Étapes de raisonnement
                        f.write("   🧠 ÉTAPES DE RAISONNEMENT:\n")
                        for k, step in enumerate(question['etapes_raisonnement'], 1):
                            f.write(f"   {k}. {step}\n")
                        f.write("\n")
                
                else:
                    f.write("⚠️  Aucune question générée pour cette paire\n\n")
                
                f.write("\n" + "="*60 + "\n\n")
        
        print(f"✅ Fichier texte élégant créé: {output_path}")
        print(f"📊 {len(results)} paires et {total_questions} questions formatées")
        
        return output_path
        
    except Exception as e:
        print(f"❌ Erreur lors de la conversion: {str(e)}")
        return None

In [23]:
def convert_jsonl_to_markdown(jsonl_file="multihop_questions_adjacent_pairs.jsonl", 
                             output_file="questions_multihop.md"):
    """
    Convertit le fichier JSONL en format Markdown élégant.
    
    Args:
        jsonl_file: Fichier JSONL source
        output_file: Fichier Markdown de sortie
    """
    import json
    from datetime import datetime
    
    try:
        input_path = os.path.join(os.getcwd(), "data", jsonl_file)
        output_path = os.path.join(os.getcwd(), "data", output_file)
        
        if not os.path.exists(input_path):
            print(f"❌ Fichier {input_path} non trouvé")
            return
        
        # Charger les données
        results = []
        with open(input_path, 'r', encoding='utf-8') as f:
            for line in f:
                results.append(json.loads(line.strip()))
        
        # Créer le fichier Markdown
        with open(output_path, 'w', encoding='utf-8') as f:
            # En-tête du document
            f.write("# 🎯 Questions Multi-Hop - Débats Parlementaires\n\n")
            f.write(f"**📅 Généré le:** {datetime.now().strftime('%d/%m/%Y à %H:%M:%S')}  \n")
            f.write(f"**📊 Nombre total de paires:** {len(results)}  \n")
            total_questions = sum(len(r['questions']) for r in results)
            f.write(f"**📝 Nombre total de questions:** {total_questions}  \n\n")
            f.write("---\n\n")
            
            # Table des matières
            f.write("## 📚 Table des Matières\n\n")
            for result in results:
                anchor = f"paire-{result['pair_id']}"
                files_display = " & ".join([f"`{file}`" for file in result['file_pair']])
                f.write(f"- [Paire {result['pair_id']}: {files_display}](#{anchor})\n")
            f.write("\n---\n\n")
            
            # Pour chaque paire
            for result in results:
                anchor = f"paire-{result['pair_id']}"
                f.write(f"## 📋 Paire {result['pair_id']}: {' & '.join(result['file_pair'])}\n\n")
                f.write(f"**🕒 Généré le:** {result['timestamp'][:19].replace('T', ' à ')}  \n")
                f.write(f"**📝 Nombre de questions:** {result['num_questions']}  \n\n")
                
                # Résumé des sources
                if result.get('resume_sources'):
                    f.write("### 📄 Résumé des Sources\n\n")
                    f.write(f"> {result['resume_sources']}\n\n")
                
                # Questions
                if result['questions']:
                    f.write("### ❓ Questions Générées\n\n")
                    
                    for j, question in enumerate(result['questions'], 1):
                        f.write(f"#### 🔹 Question {j}\n\n")
                        
                        # Métadonnées de la question
                        f.write("| Propriété | Valeur |\n")
                        f.write("|-----------|--------|\n")
                        f.write(f"| **Type** | `{question['type_question']}` |\n")
                        f.write(f"| **Difficulté** | `{question['difficulte']}` |\n")
                        f.write(f"| **Fichiers requis** | {', '.join([f'`{f}`' for f in question['fichiers_requis']])} |\n\n")
                        
                        # Question principale
                        f.write("**💭 Question:**\n\n")
                        f.write(f"> {question['question']}\n\n")
                        
                        # Étapes de raisonnement
                        f.write("**🧠 Étapes de raisonnement:**\n\n")
                        for k, step in enumerate(question['etapes_raisonnement'], 1):
                            f.write(f"{k}. {step}\n")
                        f.write("\n")
                
                else:
                    f.write("### ⚠️ Aucune question générée\n\n")
                    f.write("Aucune question n'a pu être générée pour cette paire de fichiers.\n\n")
                
                f.write("---\n\n")
        
        print(f"✅ Fichier Markdown créé: {output_path}")
        print(f"📊 {len(results)} paires et {total_questions} questions formatées")
        
        return output_path
        
    except Exception as e:
        print(f"❌ Erreur lors de la conversion: {str(e)}")
        return None

In [24]:
def convert_jsonl_to_html(jsonl_file="multihop_questions_adjacent_pairs.jsonl", 
                         output_file="questions_multihop.html"):
    """
    Convertit le fichier JSONL en format HTML élégant avec CSS.
    
    Args:
        jsonl_file: Fichier JSONL source
        output_file: Fichier HTML de sortie
    """
    import json
    from datetime import datetime
    
    try:
        input_path = os.path.join(os.getcwd(), "data", jsonl_file)
        output_path = os.path.join(os.getcwd(), "data", output_file)
        
        if not os.path.exists(input_path):
            print(f"❌ Fichier {input_path} non trouvé")
            return
        
        # Charger les données
        results = []
        with open(input_path, 'r', encoding='utf-8') as f:
            for line in f:
                results.append(json.loads(line.strip()))
        
        total_questions = sum(len(r['questions']) for r in results)
        
        # Créer le fichier HTML
        with open(output_path, 'w', encoding='utf-8') as f:
            # En-tête HTML avec CSS
            f.write('''<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Questions Multi-Hop - Débats Parlementaires</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            line-height: 1.6;
            color: #333;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f5f5;
        }
        .header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            border-radius: 10px;
            text-align: center;
            margin-bottom: 30px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .stats {
            display: flex;
            justify-content: center;
            gap: 30px;
            margin-top: 20px;
        }
        .stat-item {
            background: rgba(255,255,255,0.2);
            padding: 10px 20px;
            border-radius: 5px;
        }
        .pair-container {
            background: white;
            margin: 20px 0;
            border-radius: 10px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            overflow: hidden;
        }
        .pair-header {
            background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
            color: white;
            padding: 20px;
        }
        .pair-content {
            padding: 20px;
        }
        .source-summary {
            background: #f8f9fa;
            border-left: 4px solid #007bff;
            padding: 15px;
            margin: 15px 0;
            border-radius: 0 5px 5px 0;
        }
        .question-item {
            border: 1px solid #e9ecef;
            border-radius: 8px;
            margin: 15px 0;
            overflow: hidden;
        }
        .question-header {
            background: #f8f9fa;
            padding: 15px;
            border-bottom: 1px solid #e9ecef;
        }
        .question-meta {
            display: flex;
            gap: 15px;
            margin-top: 10px;
        }
        .meta-badge {
            background: #6c757d;
            color: white;
            padding: 4px 8px;
            border-radius: 12px;
            font-size: 0.8em;
        }
        .meta-badge.type { background: #28a745; }
        .meta-badge.difficulty { background: #ffc107; color: #000; }
        .meta-badge.files { background: #17a2b8; }
        .question-content {
            padding: 15px;
        }
        .question-text {
            font-size: 1.1em;
            font-weight: 500;
            color: #2c3e50;
            margin-bottom: 15px;
        }
        .reasoning-steps {
            background: #f8f9fa;
            padding: 15px;
            border-radius: 5px;
        }
        .reasoning-steps ol {
            margin: 0;
            padding-left: 20px;
        }
        .reasoning-steps li {
            margin: 8px 0;
        }
        .no-questions {
            text-align: center;
            padding: 30px;
            color: #6c757d;
            font-style: italic;
        }
        .toc {
            background: white;
            padding: 20px;
            border-radius: 10px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .toc ul {
            list-style: none;
            padding: 0;
        }
        .toc li {
            padding: 5px 0;
        }
        .toc a {
            text-decoration: none;
            color: #007bff;
        }
        .toc a:hover {
            text-decoration: underline;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>🎯 Questions Multi-Hop</h1>
        <h2>Débats Parlementaires de la Troisième République</h2>
        <div class="stats">
            <div class="stat-item">
                <strong>📅 Généré le</strong><br>
''')
            f.write(f'                {datetime.now().strftime("%d/%m/%Y à %H:%M:%S")}\n')
            f.write(f'''            </div>
            <div class="stat-item">
                <strong>📊 Paires</strong><br>
                {len(results)}
            </div>
            <div class="stat-item">
                <strong>📝 Questions</strong><br>
                {total_questions}
            </div>
        </div>
    </div>

    <div class="toc">
        <h3>📚 Navigation</h3>
        <ul>
''')
            
            # Table des matières
            for result in results:
                files_display = " & ".join(result['file_pair'])
                f.write(f'            <li><a href="#pair-{result["pair_id"]}">Paire {result["pair_id"]}: {files_display}</a></li>\n')
            
            f.write('''        </ul>
    </div>

''')
            
            # Pour chaque paire
            for result in results:
                f.write(f'''    <div class="pair-container" id="pair-{result["pair_id"]}">
        <div class="pair-header">
            <h2>📋 Paire {result["pair_id"]}: {" & ".join(result["file_pair"])}</h2>
            <p>🕒 Généré le: {result["timestamp"][:19].replace("T", " à ")} | 📝 {result["num_questions"]} question(s)</p>
        </div>
        <div class="pair-content">
''')
                
                # Résumé des sources
                if result.get('resume_sources'):
                    f.write(f'''            <div class="source-summary">
                <h4>📄 Résumé des Sources</h4>
                <p>{result["resume_sources"]}</p>
            </div>
''')
                
                # Questions
                if result['questions']:
                    f.write('            <h3>❓ Questions Générées</h3>\n')
                    
                    for j, question in enumerate(result['questions'], 1):
                        f.write(f'''            <div class="question-item">
                <div class="question-header">
                    <h4>🔹 Question {j}</h4>
                    <div class="question-meta">
                        <span class="meta-badge type">{question["type_question"]}</span>
                        <span class="meta-badge difficulty">{question["difficulte"]}</span>
                        <span class="meta-badge files">{", ".join(question["fichiers_requis"])}</span>
                    </div>
                </div>
                <div class="question-content">
                    <div class="question-text">
                        💭 {question["question"]}
                    </div>
                    <div class="reasoning-steps">
                        <h5>🧠 Étapes de raisonnement:</h5>
                        <ol>
''')
                        for step in question['etapes_raisonnement']:
                            f.write(f'                            <li>{step}</li>\n')
                        
                        f.write('''                        </ol>
                    </div>
                </div>
            </div>
''')
                
                else:
                    f.write('''            <div class="no-questions">
                <p>⚠️ Aucune question générée pour cette paire</p>
            </div>
''')
                
                f.write('''        </div>
    </div>

''')
            
            # Fermeture HTML
            f.write('''</body>
</html>''')
        
        print(f"✅ Fichier HTML créé: {output_path}")
        print(f"📊 {len(results)} paires et {total_questions} questions formatées")
        print(f"🌐 Ouvrez le fichier dans un navigateur pour une visualisation élégante")
        
        return output_path
        
    except Exception as e:
        print(f"❌ Erreur lors de la conversion: {str(e)}")
        return None

In [25]:
def generate_all_visual_formats(jsonl_file="multihop_questions_adjacent_pairs.jsonl"):
    """
    Génère tous les formats visuels en une seule fois.
    
    Args:
        jsonl_file: Fichier JSONL source
    """
    print("🚀 GÉNÉRATION DE TOUS LES FORMATS VISUELS")
    print("="*50)
    
    # Format texte lisible
    print("\n📝 Génération du format texte...")
    text_file = convert_jsonl_to_readable_text(jsonl_file)
    
    # Format Markdown
    print("\n📄 Génération du format Markdown...")
    md_file = convert_jsonl_to_markdown(jsonl_file)
    
    # Format HTML
    print("\n🌐 Génération du format HTML...")
    html_file = convert_jsonl_to_html(jsonl_file)
    
    print("\n" + "="*50)
    print("✅ TOUS LES FORMATS GÉNÉRÉS AVEC SUCCÈS!")
    print("="*50)
    
    if text_file:
        print(f"📝 Texte lisible: {text_file}")
    if md_file:
        print(f"📄 Markdown: {md_file}")
    if html_file:
        print(f"🌐 HTML: {html_file}")
    
    print("\n💡 RECOMMANDATIONS:")
    print("   • Utilisez le fichier HTML pour une visualisation interactive dans un navigateur")
    print("   • Utilisez le fichier Markdown pour l'intégration dans des documents")
    print("   • Utilisez le fichier texte pour une lecture simple et impression")
    
    return {
        'text': text_file,
        'markdown': md_file,
        'html': html_file
    }

In [26]:
# Génération de tous les formats visuels
print("🎨 CONVERSION EN FORMATS VISUELS ÉLÉGANTS")

# Générer tous les formats
visual_files = generate_all_visual_formats("multihop_questions_adjacent_pairs.jsonl")

🎨 CONVERSION EN FORMATS VISUELS ÉLÉGANTS
🚀 GÉNÉRATION DE TOUS LES FORMATS VISUELS

📝 Génération du format texte...
✅ Fichier texte élégant créé: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop_readable.txt
📊 13 paires et 11 questions formatées

📄 Génération du format Markdown...
✅ Fichier Markdown créé: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop.md
📊 13 paires et 11 questions formatées

🌐 Génération du format HTML...
✅ Fichier HTML créé: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop.html
📊 13 paires et 11 questions formatées
🌐 Ouvrez le fichier dans un navigateur pour une visualisation élégante

✅ TOUS LES FORMATS GÉNÉRÉS AVEC SUCCÈS!
📝 Texte lisible: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop_readable.txt
📄 Markdown: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop.md
🌐 HTML: /home/atom/Bureau/Aurelien/RAGbattre/data/questions_multihop.html

💡 RECOMMANDATIONS:
   • Utilisez le fichier HTML pour une visualisati

In [ ]:
# Exemples d'utilisation pour générer des formats spécifiques

# Générer seulement le format HTML (recommandé pour la visualisation)
# html_file = convert_jsonl_to_html("multihop_questions_adjacent_pairs.jsonl")

# Générer seulement le format Markdown
# md_file = convert_jsonl_to_markdown("multihop_questions_adjacent_pairs.jsonl")

# Générer seulement le format texte
# text_file = convert_jsonl_to_readable_text("multihop_questions_adjacent_pairs.jsonl")

print("💡 Formats disponibles:")
print("   🌐 HTML: Parfait pour la visualisation interactive dans un navigateur")
print("   📄 Markdown: Idéal pour la documentation et les rapports")
print("   📝 Texte: Simple et lisible, parfait pour l'impression")